## Análisis Multivariable

* Luisa Merlo García | A01067715
* Edgar Osvaldo Navarro | A01644488
* Ariana Guadalupe Rosales Villalobos | A01644773


## Ejercicio 3
![](foto.png)

En esta tabla, A representa los dos últimos dígitos de la matrícula de uno de los integrantes del equipo, B es el antepenúltimo dígito de la matrícula de uno de los integrantes del equipo, y C es el día de la semana favorito (del 1 al 7, comenzando por el domingo) de otro de los integrantes del equipo.

* Desarrolle tres iteraciones del método de imputación múltiple por ecuaciones encadenadas (MICE) para encontrar los datos faltantes. Los modelos de regresión de cada paso los puedes obtener con Python con cualquier librería.
* Pruebe el algoritmo de imputación múltiple disponible en la librería Scikit-learn con esta tabla, y compare los resultados obtenidos con los que tuviste en tu desarrollo.


#### Comparación

In [2]:
import warnings
import numpy as np

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression
from sklearn.exceptions import ConvergenceWarning

np.set_printoptions(precision=4, suppress=True)

X = np.array([
    [4,      3,      np.nan],
    [2,      5,      45],
    [1,      2.5,    15],
    [np.nan, 5,      70],
    [4,      np.nan, 92],
    [7,      12,     np.nan]
], dtype=float)

# Desarrollo de las tres iteraciones.
faltantes = np.isnan(X)
Y = np.where(faltantes, np.nanmean(X, axis=0), X)

for iteracion in range(1, 4):
    for j in range(3):
        observados = ~faltantes[:, j]
        predictores = np.arange(3) != j

        modelo = LinearRegression()
        modelo.fit(
            Y[observados][:, predictores],
            X[observados, j]
        )

        Y[faltantes[:, j], j] = modelo.predict(
            Y[faltantes[:, j]][:, predictores]
        )

        print(
            f"Iteración {iteracion}, variable x{j+1}:",
            "intercepto =", modelo.intercept_,
            "coeficientes =", modelo.coef_
        )

    print("Tabla al finalizar la iteración:")
    print(Y)

# tol=0 fuerza las tres rondas completas
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)

    imputador_lineal = IterativeImputer(
        estimator=LinearRegression(),
        max_iter=3,
        tol=0,
        initial_strategy="mean",
        imputation_order="roman",
        sample_posterior=False,
        random_state=0
    )
    resultado_lineal = imputador_lineal.fit_transform(X)


print("\nScikit-learn con regresión lineal:")
print(resultado_lineal)


Iteración 1, variable x1: intercepto = -0.31288512931163437 coeficientes = [0.4577 0.0257]
Iteración 1, variable x2: intercepto = 2.001519892611954 coeficientes = [ 1.7251 -0.0546]
Iteración 1, variable x3: intercepto = -3.206100562140051 coeficientes = [22.6239 -0.542 ]
Tabla al finalizar la iteración:
[[  4.       3.      85.6634]
 [  2.       5.      45.    ]
 [  1.       2.5     15.    ]
 [  3.7718   5.      70.    ]
 [  4.       3.8759  92.    ]
 [  7.      12.     148.6568]]
Iteración 2, variable x1: intercepto = 0.08342828716143913 coeficientes = [0.0418 0.0427]
Iteración 2, variable x2: intercepto = 0.49593119850877176 coeficientes = [ 2.2902 -0.0399]
Iteración 2, variable x3: intercepto = -16.269285214439677 coeficientes = [20.7596  3.9875]
Tabla al finalizar la iteración:
[[  4.       3.      78.7315]
 [  2.       5.      45.    ]
 [  1.       2.5     15.    ]
 [  3.2786   5.      70.    ]
 [  4.       5.9822  92.    ]
 [  7.      12.     176.8975]]
Iteración 3, variable x1: 